# Sycophancy Benchmark Analysis

This notebook visualizes and compares sycophancy benchmark results across multiple language models.
It analyzes per-subject and overall performance metrics from JSON report files.

## 1. Import Required Libraries

In [ ]:
import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style for matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 2. Load JSON Data Files

In [ ]:
def load_json_reports(directory: str) -> dict:
    """
    Load all JSON report files from a directory.
    
    Args:
        directory: Path to directory containing JSON report files
        
    Returns:
        Dictionary mapping model names to their report data
    """
    reports = {}
    
    # Find all JSON files matching the pattern
    pattern = os.path.join(directory, '*_syco_bench_assertion_report.json')
    json_files = glob.glob(pattern)
    
    print(f"Found {len(json_files)} JSON report files in {directory}")
    
    for filepath in json_files:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Extract model name and create a shorter display name
        model_full = data.get('model', os.path.basename(filepath))
        # Create short name (e.g., "Llama-3.2-3B" from "meta-llama/Llama-3.2-3B-Instruct")
        model_short = model_full.split('/')[-1].replace('-Instruct', '')
        
        reports[model_short] = data
        print(f"  Loaded: {model_short}")
    
    return reports

# Load reports from the per_subject directory
REPORTS_DIR = 'json_outputs/per_subject'
reports = load_json_reports(REPORTS_DIR)

print(f"\nLoaded {len(reports)} model reports: {list(reports.keys())}")

## 3. Parse and Structure Data for Comparison

In [ ]:
def extract_overall_scores(reports: dict) -> pd.DataFrame:
    """
    Extract overall scores from all models into a DataFrame.
    """
    rows = []
    for model_name, data in reports.items():
        overall = data['scores']['overall']
        rows.append({
            'model': model_name,
            'sycophant_with_knowledge': overall.get('sycophant_with_knowledge', 0),
            'agreement_rate': overall.get('agreement_rate', 0),
            'confident_sycophancy': overall.get('confident_sycophancy', 0),
        })
    return pd.DataFrame(rows).set_index('model')


def extract_per_subject_scores(reports: dict, metric: str) -> pd.DataFrame:
    """
    Extract per-subject scores for a specific metric across all models.
    
    Args:
        reports: Dictionary of model reports
        metric: One of 'sycophant_with_knowledge', 'agreement_rate', 'confident_sycophancy'
    
    Returns:
        DataFrame with subjects as rows and models as columns
    """
    data_dict = {}
    
    for model_name, report in reports.items():
        per_subject = report['scores']['per_subject'].get(metric, {})
        data_dict[model_name] = per_subject
    
    df = pd.DataFrame(data_dict)
    df = df.fillna(0)  # Fill missing subjects with 0
    df = df.sort_index()  # Sort subjects alphabetically
    
    return df


# Extract overall scores
overall_df = extract_overall_scores(reports)
print("Overall Scores:")
display(overall_df)

# Extract per-subject scores for each metric
metrics = ['sycophant_with_knowledge', 'agreement_rate', 'confident_sycophancy']
per_subject_dfs = {}

for metric in metrics:
    per_subject_dfs[metric] = extract_per_subject_scores(reports, metric)
    print(f"\nPer-subject scores for {metric}:")
    display(per_subject_dfs[metric].head(10))

## 4. Create Bar Charts for Overall Metrics

In [ ]:
def plot_overall_comparison(overall_df: pd.DataFrame):
    """
    Create grouped bar charts comparing overall sycophancy metrics across models.
    """
    # Prepare data for plotting
    df_melted = overall_df.reset_index().melt(
        id_vars='model', 
        var_name='metric', 
        value_name='score'
    )
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    metric_titles = {
        'sycophant_with_knowledge': 'Sycophant with Knowledge',
        'agreement_rate': 'Agreement Rate',
        'confident_sycophancy': 'Confident Sycophancy'
    }
    
    colors = sns.color_palette('husl', n_colors=len(overall_df))
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        values = overall_df[metric].values
        models = overall_df.index.tolist()
        
        bars = ax.bar(models, values, color=colors)
        ax.set_title(metric_titles[metric], fontsize=12, fontweight='bold')
        ax.set_ylabel('Score')
        ax.set_xlabel('Model')
        ax.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('images/overall_metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: images/overall_metrics_comparison.png")


# Create output directory if needed
os.makedirs('images', exist_ok=True)

# Plot overall comparison
if len(reports) > 0:
    plot_overall_comparison(overall_df)
else:
    print("No reports loaded. Please check the directory path.")

## 5. Create Heatmaps for Per-Subject Scores

In [ ]:
def plot_heatmap(df: pd.DataFrame, metric: str, figsize=(14, 20)):
    """
    Create a heatmap for per-subject scores.
    
    Args:
        df: DataFrame with subjects as rows and models as columns
        metric: Name of the metric for the title
        figsize: Figure size tuple
    """
    metric_titles = {
        'sycophant_with_knowledge': 'Sycophant with Knowledge',
        'agreement_rate': 'Agreement Rate',
        'confident_sycophancy': 'Confident Sycophancy'
    }
    
    plt.figure(figsize=figsize)
    
    # Create heatmap
    ax = sns.heatmap(
        df,
        annot=True,
        fmt='.2f',
        cmap='YlOrRd',
        linewidths=0.5,
        cbar_kws={'label': 'Score'},
        annot_kws={'size': 8}
    )
    
    plt.title(f'Per-Subject Scores: {metric_titles.get(metric, metric)}', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('Subject', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    filename = f'images/heatmap_{metric}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")


# Plot heatmaps for each metric
for metric in metrics:
    if not per_subject_dfs[metric].empty:
        plot_heatmap(per_subject_dfs[metric], metric)

## 6. Create Grouped Bar Charts for Subject Comparison

In [ ]:
def create_interactive_subject_comparison(per_subject_dfs: dict):
    """
    Create interactive grouped bar charts using Plotly for subject comparison.
    """
    for metric, df in per_subject_dfs.items():
        if df.empty:
            continue
            
        metric_titles = {
            'sycophant_with_knowledge': 'Sycophant with Knowledge',
            'agreement_rate': 'Agreement Rate',
            'confident_sycophancy': 'Confident Sycophancy'
        }
        
        # Reshape for plotly
        df_reset = df.reset_index().rename(columns={'index': 'subject'})
        df_melted = df_reset.melt(
            id_vars='subject',
            var_name='model',
            value_name='score'
        )
        
        # Create grouped bar chart
        fig = px.bar(
            df_melted,
            x='subject',
            y='score',
            color='model',
            barmode='group',
            title=f'Per-Subject Comparison: {metric_titles.get(metric, metric)}',
            labels={'score': 'Score', 'subject': 'Subject', 'model': 'Model'},
            height=600
        )
        
        fig.update_layout(
            xaxis_tickangle=-45,
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
            margin=dict(b=150)
        )
        
        fig.show()
        
        # Save as HTML
        filename = f'images/interactive_{metric}.html'
        fig.write_html(filename)
        print(f"Saved: {filename}")


# Create interactive charts
create_interactive_subject_comparison(per_subject_dfs)

In [ ]:
def plot_top_subjects_comparison(per_subject_dfs: dict, top_n: int = 15):
    """
    Plot bar charts for subjects with highest variance across models.
    This highlights subjects where models differ the most.
    """
    for metric, df in per_subject_dfs.items():
        if df.empty or len(df.columns) < 2:
            continue
        
        metric_titles = {
            'sycophant_with_knowledge': 'Sycophant with Knowledge',
            'agreement_rate': 'Agreement Rate',
            'confident_sycophancy': 'Confident Sycophancy'
        }
        
        # Calculate variance across models for each subject
        subject_variance = df.var(axis=1).sort_values(ascending=False)
        top_subjects = subject_variance.head(top_n).index.tolist()
        
        # Filter to top subjects
        df_top = df.loc[top_subjects]
        
        # Create plot
        fig, ax = plt.subplots(figsize=(14, 8))
        
        x = np.arange(len(top_subjects))
        width = 0.8 / len(df.columns)
        colors = sns.color_palette('husl', n_colors=len(df.columns))
        
        for i, model in enumerate(df.columns):
            offset = (i - len(df.columns)/2 + 0.5) * width
            ax.bar(x + offset, df_top[model].values, width, label=model, color=colors[i])
        
        ax.set_xlabel('Subject', fontsize=12)
        ax.set_ylabel('Score', fontsize=12)
        ax.set_title(f'Top {top_n} Subjects with Highest Model Variance: {metric_titles.get(metric, metric)}',
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(top_subjects, rotation=45, ha='right')
        ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
        
        plt.tight_layout()
        filename = f'images/top_variance_subjects_{metric}.png'
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {filename}")


# Plot top subjects by variance
plot_top_subjects_comparison(per_subject_dfs)

## 7. Create Radar Charts for Multi-Model Comparison

In [ ]:
def create_radar_chart(per_subject_dfs: dict, subjects_to_include: list = None, metric: str = 'confident_sycophancy'):
    """
    Create radar/spider chart comparing models across selected subjects.
    
    Args:
        per_subject_dfs: Dictionary of DataFrames per metric
        subjects_to_include: List of subjects to include (uses top variance if None)
        metric: Which metric to visualize
    """
    df = per_subject_dfs[metric]
    
    if df.empty:
        print(f"No data for metric: {metric}")
        return
    
    # Select subjects with highest mean scores if not specified
    if subjects_to_include is None:
        subject_means = df.mean(axis=1).sort_values(ascending=False)
        subjects_to_include = subject_means.head(10).index.tolist()
    
    df_subset = df.loc[subjects_to_include]
    
    # Create radar chart
    fig = go.Figure()
    
    for model in df.columns:
        values = df_subset[model].tolist()
        values.append(values[0])  # Close the polygon
        
        categories = subjects_to_include.copy()
        categories.append(categories[0])
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories,
            fill='toself',
            name=model,
            opacity=0.6
        ))
    
    metric_titles = {
        'sycophant_with_knowledge': 'Sycophant with Knowledge',
        'agreement_rate': 'Agreement Rate',
        'confident_sycophancy': 'Confident Sycophancy'
    }
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, max(df_subset.max()) * 1.1]
            )
        ),
        showlegend=True,
        title=f'Radar Chart: {metric_titles.get(metric, metric)}',
        height=700
    )
    
    fig.show()
    
    filename = f'images/radar_{metric}.html'
    fig.write_html(filename)
    print(f"Saved: {filename}")


# Create radar charts for each metric
for metric in metrics:
    create_radar_chart(per_subject_dfs, metric=metric)

In [ ]:
# Create a radar chart with custom subject selection
# You can modify this list to include subjects of interest
custom_subjects = [
    'machine_learning',
    'professional_law',
    'business_ethics',
    'high_school_biology',
    'elementary_mathematics',
    'college_physics',
    'philosophy',
    'computer_security'
]

# Filter to only include subjects that exist in the data
available_subjects = [s for s in custom_subjects if s in per_subject_dfs['confident_sycophancy'].index]

if available_subjects:
    create_radar_chart(per_subject_dfs, subjects_to_include=available_subjects, metric='confident_sycophancy')

## 8. Generate Summary Statistics Table

In [ ]:
def generate_summary_statistics(per_subject_dfs: dict) -> pd.DataFrame:
    """
    Generate summary statistics for each model across subjects.
    
    Returns:
        DataFrame with statistics for each model and metric
    """
    summary_rows = []
    
    for metric, df in per_subject_dfs.items():
        for model in df.columns:
            values = df[model]
            summary_rows.append({
                'metric': metric,
                'model': model,
                'mean': values.mean(),
                'std': values.std(),
                'min': values.min(),
                'max': values.max(),
                'median': values.median(),
                'non_zero_count': (values > 0).sum(),
                'total_subjects': len(values)
            })
    
    summary_df = pd.DataFrame(summary_rows)
    return summary_df


# Generate and display summary statistics
summary_df = generate_summary_statistics(per_subject_dfs)

# Format for display
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

for metric in metrics:
    metric_titles = {
        'sycophant_with_knowledge': 'Sycophant with Knowledge',
        'agreement_rate': 'Agreement Rate',
        'confident_sycophancy': 'Confident Sycophancy'
    }
    print(f"\n{metric_titles.get(metric, metric)}:")
    display(summary_df[summary_df['metric'] == metric].drop(columns='metric').set_index('model').round(4))

In [ ]:
def create_styled_summary_table(summary_df: pd.DataFrame, metric: str):
    """
    Create a styled HTML table for a specific metric.
    """
    metric_df = summary_df[summary_df['metric'] == metric].drop(columns='metric').set_index('model')
    
    # Style the table
    styled = metric_df.style\
        .format({
            'mean': '{:.4f}',
            'std': '{:.4f}',
            'min': '{:.4f}',
            'max': '{:.4f}',
            'median': '{:.4f}',
        })\
        .background_gradient(subset=['mean'], cmap='YlOrRd')\
        .background_gradient(subset=['std'], cmap='Blues')\
        .set_caption(f'Summary Statistics: {metric}')
    
    return styled


# Display styled tables
for metric in metrics:
    display(create_styled_summary_table(summary_df, metric))

In [ ]:
# Save summary statistics to CSV
summary_df.to_csv('images/summary_statistics.csv', index=False)
print("Saved: images/summary_statistics.csv")

# Also save overall scores
overall_df.to_csv('images/overall_scores.csv')
print("Saved: images/overall_scores.csv")

## Additional Analysis: Subject Category Grouping

In [ ]:
# Define subject categories for grouped analysis
SUBJECT_CATEGORIES = {
    'STEM': [
        'abstract_algebra', 'astronomy', 'college_biology', 'college_chemistry',
        'college_computer_science', 'college_mathematics', 'college_physics',
        'computer_security', 'conceptual_physics', 'electrical_engineering',
        'elementary_mathematics', 'high_school_biology', 'high_school_chemistry',
        'high_school_computer_science', 'high_school_mathematics', 'high_school_physics',
        'high_school_statistics', 'machine_learning'
    ],
    'Humanities': [
        'high_school_european_history', 'high_school_us_history', 'high_school_world_history',
        'philosophy', 'prehistory', 'world_religions', 'formal_logic', 'logical_fallacies'
    ],
    'Social Sciences': [
        'business_ethics', 'econometrics', 'high_school_geography',
        'high_school_government_and_politics', 'high_school_macroeconomics',
        'high_school_microeconomics', 'high_school_psychology', 'human_sexuality',
        'international_law', 'jurisprudence', 'management', 'marketing',
        'moral_disputes', 'moral_scenarios', 'public_relations', 'security_studies',
        'sociology', 'us_foreign_policy'
    ],
    'Medical/Health': [
        'anatomy', 'clinical_knowledge', 'college_medicine', 'human_aging',
        'medical_genetics', 'nutrition', 'professional_medicine', 'virology'
    ],
    'Professional': [
        'professional_accounting', 'professional_law', 'professional_psychology'
    ],
    'Other': [
        'global_facts', 'miscellaneous'
    ]
}


def analyze_by_category(per_subject_dfs: dict, metric: str) -> pd.DataFrame:
    """
    Compute average scores by subject category.
    """
    df = per_subject_dfs[metric]
    
    category_scores = {}
    for category, subjects in SUBJECT_CATEGORIES.items():
        # Filter to subjects that exist in the data
        valid_subjects = [s for s in subjects if s in df.index]
        if valid_subjects:
            category_scores[category] = df.loc[valid_subjects].mean()
    
    return pd.DataFrame(category_scores).T


# Analyze by category
print("\n" + "="*80)
print("ANALYSIS BY SUBJECT CATEGORY")
print("="*80)

for metric in metrics:
    metric_titles = {
        'sycophant_with_knowledge': 'Sycophant with Knowledge',
        'agreement_rate': 'Agreement Rate',
        'confident_sycophancy': 'Confident Sycophancy'
    }
    print(f"\n{metric_titles.get(metric, metric)} - Average by Category:")
    category_df = analyze_by_category(per_subject_dfs, metric)
    display(category_df.round(4))

In [ ]:
def plot_category_comparison(per_subject_dfs: dict, metric: str = 'confident_sycophancy'):
    """
    Create bar chart comparing models by subject category.
    """
    category_df = analyze_by_category(per_subject_dfs, metric)
    
    if category_df.empty:
        return
    
    metric_titles = {
        'sycophant_with_knowledge': 'Sycophant with Knowledge',
        'agreement_rate': 'Agreement Rate',
        'confident_sycophancy': 'Confident Sycophancy'
    }
    
    # Create grouped bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    
    categories = category_df.index.tolist()
    x = np.arange(len(categories))
    width = 0.8 / len(category_df.columns)
    colors = sns.color_palette('husl', n_colors=len(category_df.columns))
    
    for i, model in enumerate(category_df.columns):
        offset = (i - len(category_df.columns)/2 + 0.5) * width
        ax.bar(x + offset, category_df[model].values, width, label=model, color=colors[i])
    
    ax.set_xlabel('Subject Category', fontsize=12)
    ax.set_ylabel('Average Score', fontsize=12)
    ax.set_title(f'Average {metric_titles.get(metric, metric)} by Subject Category',
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=45, ha='right')
    ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    
    plt.tight_layout()
    filename = f'images/category_comparison_{metric}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")


# Plot category comparisons
for metric in metrics:
    plot_category_comparison(per_subject_dfs, metric)